# Flight Satisfaction — CatBoost K-Fold Ensemble

Clean, self-contained pipeline:
1. **Load** `train.csv` / `test.csv`
2. **Median imputation** via `SimpleImputer` fitted on train only
3. **Outlier capping** — 99th-percentile on delay columns
4. **Feature engineering** — delay aggregates, service composite scores, interaction features
5. **5-fold Stratified K-Fold CatBoost ensemble** (`iterations=3000`, `lr=0.05`, `depth=8`, `eval_metric=AUC`)
   — raw string categoricals passed directly to CatBoost, no label-encoding
6. **Save** averaged test probabilities to `result_with_cat_boost.csv`

## 1. Load Data

In [14]:
df = pd.read_csv('train.csv')
dt = pd.read_csv("data.csv")

df = df.drop(columns=["id"])
df = pd.concat([df, dt], ignore_index=True)
df

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,36,694,4,4,4,3,1,4,1,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,56,651,5,5,5,5,5,4,4,5,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,31,1371,3,4,2,5,3,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,10,1616,3,5,3,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,23,481,3,4,3,4,5,3,5,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829510,34,526,3,3,3,1,4,3,4,4,...,4,4,4,0,0.0,Male,disloyal Customer,Business travel,Business,False
829511,23,646,4,4,4,4,4,4,4,4,...,5,5,4,0,0.0,Male,Loyal Customer,Business travel,Business,True
829512,17,828,2,5,1,5,2,1,2,2,...,4,5,2,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
829513,14,1127,3,3,3,3,4,4,4,4,...,5,4,4,0,0.0,Male,Loyal Customer,Business travel,Business,True


In [15]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv('data.csv')
test  = pd.read_csv('test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Missing values in train:\n{train.isnull().sum()[train.isnull().sum() > 0]}')
print(f'Missing values in test:\n{test.isnull().sum()[test.isnull().sum() > 0]}')

Train: (129880, 22)  |  Test: (299844, 22)
Missing values in train:
Arrival Delay in Minutes    393
dtype: int64
Missing values in test:
Arrival Delay in Minutes    130
dtype: int64


## 2. Preprocessing

- Drop `id`; save test IDs separately for the submission
- `SimpleImputer(strategy='median')` fitted on train numerics, applied to both train and test
- Cap delay columns at their 99th-percentile training value

In [16]:
# ── Separate IDs and target ───────────────────────────────────────────────────
test_ids = test['id'].copy()

df_train = train
df_test  = test.drop(columns=['id']).copy()

# Encode target bool → int
df_train['satisfaction'] = df_train['satisfaction'].astype(int)

# ── Median imputation on all numeric columns ──────────────────────────────────
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numeric_cols = [c for c in df_train.columns
                if c not in cat_cols and c != 'satisfaction']

imputer = SimpleImputer(strategy='median')
df_train[numeric_cols] = imputer.fit_transform(df_train[numeric_cols])
df_test[numeric_cols]  = imputer.transform(df_test[numeric_cols])

print(f'Numeric columns imputed ({len(numeric_cols)}): {numeric_cols}')
print(f'Missing after imputation — train: {df_train[numeric_cols].isnull().sum().sum()}  '
      f'test: {df_test[numeric_cols].isnull().sum().sum()}')

# ── Cap delay outliers at 99th-percentile thresholds (computed on train) ──────
delay_cols = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
caps = {}
for col in delay_cols:
    cap = df_train[col].quantile(0.99)
    caps[col] = cap
    df_train[col] = df_train[col].clip(upper=cap)
    df_test[col]  = df_test[col].clip(upper=cap)

print('99th-percentile caps applied:')
for col, cap in caps.items():
    print(f'  {col}: {cap:.1f} min')

# ── Ensure cat cols are clean strings (CatBoost requires no NaN in cat cols) ──
for col in cat_cols:
    df_train[col] = df_train[col].fillna('__MISSING__').astype(str)
    df_test[col]  = df_test[col].fillna('__MISSING__').astype(str)

Numeric columns imputed (17): ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
Missing after imputation — train: 0  test: 0
99th-percentile caps applied:
  Departure Delay in Minutes: 180.0 min
  Arrival Delay in Minutes: 182.0 min


## 3. Feature Engineering

All features computed from existing columns — same formula applied to train and test.

| Group | Features |
|---|---|
| Delay | `total_delay`, `delay_diff`, `delay_recovered`, `any_delay` |
| Service composites | `comfort_score`, `digital_score`, `inflight_exp_score`, `ground_score` |
| Interactions | `service_range`, `avg_service_rating`, `dist_per_age` |

In [17]:
def add_features(df):
    """Apply all feature engineering steps in-place and return the dataframe."""
    dep = df['Departure Delay in Minutes']
    arr = df['Arrival Delay in Minutes']

    df['total_delay']      = dep + arr
    df['delay_diff']       = arr - dep
    df['delay_recovered']  = (df['delay_diff'] < 0).astype(int)
    df['any_delay']        = ((dep > 0) | (arr > 0)).astype(int)

    comfort_cols  = ['Seat comfort', 'Leg room service']
    digital_cols  = ['Inflight wifi service', 'Online boarding', 'Ease of Online booking']
    inflight_cols = ['Inflight entertainment', 'Food and drink', 'Cleanliness', 'On-board service']
    ground_cols   = ['Gate location', 'Checkin service', 'Baggage handling']
    all_svc_cols  = comfort_cols + digital_cols + inflight_cols + ground_cols + \
                    ['Departure/Arrival time convenient']

    df['comfort_score']      = df[comfort_cols].mean(axis=1)
    df['digital_score']      = df[digital_cols].mean(axis=1)
    df['inflight_exp_score'] = df[inflight_cols].mean(axis=1)
    df['ground_score']       = df[ground_cols].mean(axis=1)
    df['service_range']      = df[all_svc_cols].max(axis=1) - df[all_svc_cols].min(axis=1)
    df['avg_service_rating'] = df[all_svc_cols].mean(axis=1)
    df['dist_per_age']       = df['Flight Distance'] / df['Age']

    return df

df_train = add_features(df_train)
df_test  = add_features(df_test)

print(f'Train shape after feature engineering: {df_train.shape}')
print(f'Test  shape after feature engineering: {df_test.shape}')

Train shape after feature engineering: (129880, 33)
Test  shape after feature engineering: (299844, 32)


## 4. Prepare X / y

In [18]:
X = df_train.drop(columns=['satisfaction'])
y = df_train['satisfaction']

X_test_final = df_test[X.columns]   # align column order

print(f'X: {X.shape}  |  X_test_final: {X_test_final.shape}')
print(f'Class balance: {y.value_counts(normalize=True).round(3).to_dict()}')

X: (129880, 32)  |  X_test_final: (299844, 32)
Class balance: {0: 0.566, 1: 0.434}


## 5. CatBoost 5-Fold K-Fold Ensemble

- **`StratifiedKFold`** preserves class balance in every fold
- **10 % internal hold-out** carved from each fold's training slice for early stopping — the validation fold is never touched during training (no leakage)
- Each fold's model predicts on the full test set; 5 probability arrays are **averaged**
- OOF probabilities give a clean overall AUC estimate on the training set

In [19]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

test_probas = np.zeros(len(X_test_final))
oof_probas  = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx],  y.iloc[val_idx]

    # 10 % internal hold-out for early stopping — validation fold stays clean
    n_es = int(len(X_tr) * 0.1)
    X_es, y_es = X_tr.iloc[-n_es:], y_tr.iloc[-n_es:]
    X_tr, y_tr = X_tr.iloc[:-n_es],  y_tr.iloc[:-n_es]

    model_fold = CatBoostClassifier(
        iterations=3000,
        cat_features=cat_cols,
        learning_rate=0.05,
        depth=8,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=200,
        allow_writing_files=False,
        thread_count=-1,
    )
    model_fold.fit(X_tr, y_tr, eval_set=(X_es, y_es), early_stopping_rounds=150)

    oof_probas[val_idx]  = model_fold.predict_proba(X_val)[:, 1]
    test_probas         += model_fold.predict_proba(X_test_final)[:, 1]

    fold_auc = roc_auc_score(y_val, oof_probas[val_idx])
    print(f'Fold {fold}/{N_SPLITS}  |  best iter: {model_fold.best_iteration_:>4}  |  OOF AUC: {fold_auc:.5f}')

test_probas /= N_SPLITS

overall_oof_auc = roc_auc_score(y, oof_probas)
print(f'\nOverall OOF AUC: {overall_oof_auc:.5f}')

0:	test: 0.9752267	best: 0.9752267 (0)	total: 99.3ms	remaining: 4m 57s
200:	test: 0.9940900	best: 0.9940900 (200)	total: 22.1s	remaining: 5m 8s
400:	test: 0.9946122	best: 0.9946122 (400)	total: 49.8s	remaining: 5m 22s
600:	test: 0.9947217	best: 0.9947243 (596)	total: 1m 17s	remaining: 5m 9s
800:	test: 0.9948012	best: 0.9948078 (787)	total: 1m 37s	remaining: 4m 27s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9948077846
bestIteration = 787

Shrink model to first 788 iterations.
Fold 1/5  |  best iter:  787  |  OOF AUC: 0.99461
0:	test: 0.9755884	best: 0.9755884 (0)	total: 93ms	remaining: 4m 38s
200:	test: 0.9939704	best: 0.9939704 (200)	total: 19.8s	remaining: 4m 35s
400:	test: 0.9943881	best: 0.9943916 (396)	total: 39.6s	remaining: 4m 16s
600:	test: 0.9945203	best: 0.9945335 (594)	total: 1m 2s	remaining: 4m 7s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9945400271
bestIteration = 649

Shrink model to first 650 iterations.
Fold 2/5  |  be

## 6. Save Submission

In [20]:
result_with_cat_boost = pd.DataFrame({
    'id'          : test_ids.values,
    'satisfaction': test_probas,
})

result_with_cat_boost.to_csv('result_with_cat_boost.csv', index=False)

print(f'Saved result_with_cat_boost.csv  ({len(result_with_cat_boost):,} rows)')
display(result_with_cat_boost.head(10))

Saved result_with_cat_boost.csv  (299,844 rows)


,id,satisfaction
0,699635,0.327105
1,699636,0.887965
2,699637,0.920438
3,699638,0.077573
4,699639,0.000340
5,699640,0.999093
6,699641,0.000049
7,699642,0.999768
8,699643,0.999972
9,699644,0.000520
